In [2]:


import joblib

X_train = joblib.load(
    "X_train_strat.pkl"
)

X_test = joblib.load(
    "X_test_strat.pkl"
)

y_train = joblib.load(
    "y_train_strat.pkl"
)

y_test = joblib.load(
    "y_test_strat.pkl"
)

In [3]:


import numpy as np
import optuna

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    f_regression
)

from sklearn.svm import SVR

from sklearn.model_selection import cross_val_score

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

In [4]:


preprocessor = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "variance_filter",
        VarianceThreshold(threshold=0.01)
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "feature_selection",
        SelectKBest(
            score_func=f_regression,
            k=1000
        )
    )

])

In [5]:


def objective(trial):

    model = SVR(

        C=trial.suggest_float(
            "C",
            1e-2,
            1e3,
            log=True
        ),

        epsilon=trial.suggest_float(
            "epsilon",
            1e-4,
            1.0,
            log=True
        ),

        gamma=trial.suggest_categorical(
            "gamma",
            ["scale", "auto"]
        ),

        kernel="rbf"
    )

    pipeline = Pipeline([

        (
            "preprocessing",
            preprocessor
        ),

        (
            "model",
            model
        )

    ])

    scores = cross_val_score(

        pipeline,

        X_train,

        y_train,

        cv=5,

        scoring="r2",

        n_jobs=-1

    )

    return scores.mean()

In [6]:

study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=True
)

[I 2026-06-03 12:58:15,839] A new study created in memory with name: no-name-c14a4915-e24f-4774-b10e-c6e6f892b3f5


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-06-03 12:59:32,559] Trial 0 finished with value: 0.44061707665140687 and parameters: {'C': 0.013938129260056864, 'epsilon': 0.00014611471270071136, 'gamma': 'auto'}. Best is trial 0 with value: 0.44061707665140687.
[I 2026-06-03 13:00:11,019] Trial 1 finished with value: 0.7665169103410127 and parameters: {'C': 3.8852352355416726, 'epsilon': 0.6312916548365516, 'gamma': 'auto'}. Best is trial 1 with value: 0.7665169103410127.
[I 2026-06-03 13:01:28,284] Trial 2 finished with value: 0.6859222655699273 and parameters: {'C': 0.17273566411465632, 'epsilon': 0.007363877569481485, 'gamma': 'auto'}. Best is trial 1 with value: 0.7665169103410127.
[I 2026-06-03 13:02:59,703] Trial 3 finished with value: 0.7468264492395663 and parameters: {'C': 0.6240684225206585, 'epsilon': 0.00011092562495446727, 'gamma': 'scale'}. Best is trial 1 with value: 0.7665169103410127.
[I 2026-06-03 13:04:52,559] Trial 4 finished with value: 0.7173662245555706 and parameters: {'C': 615.1373165689023, 'epsilo

In [7]:


print("Best CV R2:")
print(study.best_value)

print()

print("Best Parameters:")
print(study.best_params)

Best CV R2:
0.7759044937782285

Best Parameters:
{'C': 5.939778874591412, 'epsilon': 0.13436412370110914, 'gamma': 'scale'}


In [8]:

best_svr_pipeline = Pipeline([

    (
        "preprocessing",
        preprocessor
    ),

    (
        "model",

        SVR(

            **study.best_params,

            kernel="rbf"
        )
    )

])

best_svr_pipeline.fit(
    X_train,
    y_train
)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('imputer', ...), ('variance_filter', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None


In [11]:
# TRAIN PREDICTIONS

y_train_pred = best_svr_pipeline.predict(
    X_train
)

# TEST PREDICTIONS

y_test_pred = best_svr_pipeline.predict(
    X_test
)

In [12]:


train_r2 = r2_score(
    y_train,
    y_train_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)

train_rmse = np.sqrt(
    mean_squared_error(
        y_train,
        y_train_pred
    )
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

train_mae = mean_absolute_error(
    y_train,
    y_train_pred
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

print("\n TUNED SVR ")

print("\nTrain R2 :", train_r2)
print("Train RMSE :", train_rmse)
print("Train MAE :", train_mae)

print("\nTest R2 :", test_r2)
print("Test RMSE :", test_rmse)
print("Test MAE :", test_mae)


 TUNED SVR 

Train R2 : 0.9102718561803894
Train RMSE : 0.7086320124587155
Train MAE : 0.34138914667544157

Test R2 : 0.7912294163149043
Test RMSE : 1.085980062806624
Test MAE : 0.6996377364016931


In [13]:
print("Best Parameters:")
print(study.best_params)

Best Parameters:
{'C': 5.939778874591412, 'epsilon': 0.13436412370110914, 'gamma': 'scale'}


In [ ]:
"""import joblib

joblib.dump(
    best_svr_pipeline,
    "svr_optuna_pipeline.pkl"
)